In [98]:
%config SqlMagic.autopolars = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

In [99]:
%load_ext sql

# Dépendances


In [100]:
import json
import math
import os
from datetime import datetime, timedelta
from itertools import product
from pathlib import Path
from zoneinfo import ZoneInfo
from dotenv import load_dotenv
import requests
import time

import branca.colormap as bcm
import duckdb
import folium
import geopandas as gpd
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
import polars_h3 as plh3
import polars_st as st
import shapely
from dotenv import load_dotenv
from folium import plugins
from sqlalchemy import create_engine

# Configuration


In [101]:
USE_CACHED_ARTIFACTS = True
USE_CACHED_JOURNEYS_WITH_NEAREST_STATION = True


DEFAULT_START_DATE = datetime(2024, 9, 1, tzinfo=ZoneInfo("GMT"))
ENTITY_CONFIGS = {
    "driver": {
        "identity_col": "driver_identity_key",
        "first_trip_col": "first_trip_datetime",
        "label_plural": "conducteurs",
        "label_singular": "conducteur",
    },
    "passenger": {
        "identity_col": "passenger_identity_key",
        "first_trip_col": "passenger_first_trip_datetime", 
        "label_plural": "passagers",
        "label_singular": "passager",
    }
}

In [102]:
load_dotenv()
DB_URL = os.environ.get("DB_URL")
IDF_GEOJSON = os.environ.get("IDF_GEOJSON")
BUDGET_IDFM = os.environ.get("BUDGET_IDFM")
DUREE_CAMPAGNE_IDFM = os.environ.get("DUREE_CAMPAGNE_IDFM")

In [103]:
AOM_SIRET = "28750007800020"

In [104]:
OUTPUT_PATH = Path("outputs_idfm")

## Labels

In [105]:
labels_map = {
    "month": "Mois",
    "num_journeys": "Nombre de trajets",
    "share_journeys": "% des trajets",
    "num_journeys_incentived": "Nombre de trajets avec incitation",
    "num_journeys_with_incentive": "Nombre de trajets avec incitation",
    "num_journeys_intra_territory_incentived_trips": "Nombre de trajets incités intra",
    "num_journeys_inter_territory_incentived_trips": "Nombre de trajets incités inter",
    "operator": "Opérateur",
    "incentive_amount_avg": "Incitation moyenne",
    "driver_revenue_avg": "Revenu moyen conducteur",
    "passenger_contribution_avg": "Contribution moyenne passager",
    "incentive_amount_intra_avg": "Incitation moyenne intra",
    "driver_revenue_intra_avg": "Revenu moyen conducteur intra",
    "passenger_contribution_intra_avg": "Contribution moyenne passager intra",
    "incentive_amount_inter_avg": "Incitation moyenne inter",
    "driver_revenue_inter_avg": "Revenu moyen conducteur inter",
    "passenger_contribution_inter_avg": "Contribution moyenne passager inter",
    "incentive_amount_per_km_avg": "Montant moyen d'incitation par km",
    "passenger_contribution_per_km_avg": "Contribution moyenne passager par km",
    "driver_revenue_per_km_avg": "Revenu moyen conducteur par km",
    "day": "Jour",
    "week": "Semaine",
    "month": "Mois",
    "year_month": "Mois",
    "distance_avg": "Distance moyenne",
    "distance_km": "Distance [km]",
    "distance_incentived_trips_avg": "Distance moyenne [km] - trajets avec incentives",
    "campaign_type": "Campagne",
    "distance": "Distance",
    "num_journeys_with_aom_incentive": "Nombre de trajets incités par l'AOM",
    "num_journeys_with_operator_incentive": "Nombre de trajets incités par un opérateur",
    "num_journeys_intra_territory": "Nombre de trajets intra-territoire",
    "num_journeys_inter_territory": "Nombre de trajets inter-territoires",
    "share_journeys_intra_territory": "% de trajets intra-territoire",
    "share_journeys_inter_territory": "% de trajets inter-territoires",
    "share_drivers": "% des conducteurs",
    "num_trips": "Nombre de trips",
    "is_intra_driver": "Conducteur intra",
    "driver_campaign_type": "Type de campagne du conducteur",
    "passenger_campaign_type": "Type de campagne du passager",
    "drivers_share": "% des conducteurs",
    "week_number": "Semaine n°",
    "passengers_share": "% des passagers",
    "num_passenger": "Nombre de passager",
    "is_near_station_fmt": "Catégorie de distance à une gare",
    "has_direct_train_line": "Possède une ligne TC directe",
    "name": "Nom",
    "amount_aom_avg" : "Incitation AOM moyenne",
    "line_name_end":"Ligne",
    "amount_aom": "Incitation AOM",
    "incentive_amount": "Incitation totale",
    "variable": "",
}

import plotly.io as pio

pio.templates.default = "simple_white"
pio.templates[pio.templates.default].layout.font.size = 16

In [106]:
incentived_trip_filter_expr = pl.col("incentive_amount") > 0

agg_expressions = [
    pl.col("_id").n_unique().alias("num_journeys"),
    pl.col("_id")
    .filter(pl.col("is_fully_inside_campaign_area"))
    .n_unique()
    .alias("num_journeys_intra_territory"),
    pl.col("_id")
    .filter(pl.col("is_fully_inside_campaign_area") & incentived_trip_filter_expr)
    .n_unique()
    .alias("num_journeys_intra_territory_incentived_trips"),
    pl.col("_id")
    .filter(incentived_trip_filter_expr)
    .n_unique()
    .alias("num_journeys_incentived"),
    pl.col("_id")
    .filter(pl.col("incentived_by_aom"))
    .n_unique()
    .alias("num_journeys_with_aom_incentive"),
    pl.col("_id")
    .filter(pl.col("incentived_by_operator"))
    .n_unique()
    .alias("num_journeys_with_operator_incentive"),
    pl.col("_id")
    .filter(pl.col("is_fully_inside_campaign_area"))
    .n_unique()
    .alias("num_journeys_intra"),
    (pl.col("distance") / 1000).mean().alias("distance_avg"),
    (pl.col("distance").filter(incentived_trip_filter_expr) / 1000)
    .mean()
    .alias("distance_incentived_trips_avg"),
    (pl.col("amount_aom").sum()/100).alias("amount_aom_sum"),
    (pl.col("amount_aom").mean()/100).alias("amount_aom_avg"),
    (pl.col("incentive_amount").mean() / 100).alias("incentive_amount_avg"),
    (pl.col("passenger_contribution").filter(incentived_trip_filter_expr.not_()) / 100)
    .mean()
    .alias("passenger_contribution_avg"),
    (pl.col("passenger_contribution").filter(incentived_trip_filter_expr) / 100)
    .mean()
    .alias("passenger_contribution_incentived_trips_avg"),
    (
        pl.col("driver_revenue").filter(incentived_trip_filter_expr.not_()).mean() / 100
    ).alias("driver_revenue_avg"),
    (pl.col("driver_revenue").filter(incentived_trip_filter_expr).mean() / 100).alias(
        "driver_revenue_incentived_trips_avg"
    ),
    (
        pl.col("incentive_amount")
        .filter(pl.col("is_fully_inside_campaign_area"))
        .mean()
        / 100
    ).alias("incentive_amount_intra_avg"),
    (
        pl.col("passenger_contribution")
        .filter(pl.col("is_fully_inside_campaign_area") & incentived_trip_filter_expr)
        .mean()
        / 100
    ).alias("passenger_contribution_intra_avg"),
    (
        pl.col("driver_revenue")
        .filter(pl.col("is_fully_inside_campaign_area") & incentived_trip_filter_expr)
        .mean()
        / 100
    ).alias("driver_revenue_intra_avg"),
    (
        pl.col("incentive_amount")
        .filter(
            pl.col("is_fully_inside_campaign_area").not_() & incentived_trip_filter_expr
        )
        .mean()
        / 100
    ).alias("incentive_amount_inter_avg"),
    (
        pl.col("passenger_contribution")
        .filter(
            pl.col("is_fully_inside_campaign_area").not_() & incentived_trip_filter_expr
        )
        .mean()
        / 100
    ).alias("passenger_contribution_inter_avg"),
    (
        pl.col("driver_revenue")
        .filter(
            pl.col("is_fully_inside_campaign_area").not_() & incentived_trip_filter_expr
        )
        .mean()
        / 100
    ).alias("driver_revenue_inter_avg"),
    (10 * (pl.col("incentive_amount") / pl.col("distance")))
    .mean()
    .alias("incentive_amount_per_km_avg"),
    (10 * (pl.col("amount_aom") / pl.col("distance")).filter(
            incentived_trip_filter_expr
        ))
    .mean()
    .alias("aom_amount_per_km_avg"),    
    (
        10
        * (pl.col("passenger_contribution") / pl.col("distance")).filter(
            incentived_trip_filter_expr
        )
    )
    .mean()
    .alias("passenger_contribution_per_km_avg"),
    (
        10
        * (pl.col("passenger_contribution") / pl.col("distance")).filter(
            incentived_trip_filter_expr
        )
    )
    .mean()
    .alias("passenger_contribution_per_km_incentived_trips_avg"),
    (
        10
        * (pl.col("driver_revenue") / pl.col("distance")).filter(
            incentived_trip_filter_expr
        )
    )
    .mean()
    .alias("driver_revenue_per_km_avg"),
    (
        10
        * (pl.col("driver_revenue") / pl.col("distance")).filter(
            incentived_trip_filter_expr.not_() &  pl.col("is_fully_inside_campaign_area").not_()
        )
    )
    .mean()
    .alias("driver_revenue_per_km_avg_inter"),
     (
        10
        * (pl.col("driver_revenue") / pl.col("distance")).filter(
            incentived_trip_filter_expr.not_() &  pl.col("is_fully_inside_campaign_area")
        )
    )
    .mean()
    .alias("driver_revenue_per_km_avg_intra"),
    (
        10
        * (pl.col("driver_revenue") / pl.col("distance")).filter(
            incentived_trip_filter_expr
        )
    )
    .mean()
    .alias("driver_revenue_per_km_incentived_trips_avg"),
    (
        10
        * (pl.col("driver_revenue") / pl.col("distance")).filter(
            incentived_trip_filter_expr &  pl.col("is_fully_inside_campaign_area").not_()
        )
    )
    .mean()
    .alias("driver_revenue_per_km_incentived_trips_avg_inter"),
    (
        10
        * (pl.col("driver_revenue") / pl.col("distance")).filter(
            incentived_trip_filter_expr &  pl.col("is_fully_inside_campaign_area")
        )
    )
    .mean()
    .alias("driver_revenue_per_km_incentived_trips_avg_intra"),
    pl.col("driver_identity_key").n_unique().alias("number_of_unique_driver"),
    pl.col("passenger_identity_key").n_unique().alias("number_of_unique_passenger"),

]

## duckdb


In [107]:
conn = duckdb.connect(
    "db.duckdb",
    config={"memory_limit": "16GiB", "threads": 4, "preserve_insertion_order": False},
)
%sql conn --alias duckdb

In [108]:
%%sql
INSTALL spatial;

LOAD spatial;

In [109]:
SQL_ENGINE = create_engine(DB_URL)

## Opérateurs

In [110]:
df_operators = pl.read_database(
    query="""
SELECT
    "_id",
    "name",
    "siret"
from operator.operators
where deleted_at is null
and name!='BlaBlaCar'
""",
    connection=SQL_ENGINE,
)
df_karos=pl.DataFrame({"_id":[999], "name":["Karos"], "siret":["80279897500024"]})
df_operators = pl.concat([df_operators, df_karos])
df_operators

In [111]:
df_journeys_raw = pl.read_parquet("df_journeys_raw_idfm.parquet")
df_journeys_raw = df_journeys_raw.with_columns(
    pl.col("incentive_sirets").list.contains(AOM_SIRET).alias("incentived_by_aom"),
    (
        pl.col("incentive_sirets")
        .list.set_intersection(df_operators["siret"].to_list())
        .list.len()
        > 0
    ).alias("incentived_by_operator"),
)

# Etude des utilisateurs par fréquence d'utilisation des plateformes de mise en relation pour le covoiturage

In [112]:
NUM_MONTHS = 10

def create_user_flags(df, user_type):
    """
    Créer les flags one_shot, ten_shots et regular pour un type d'utilisateur
    user_type: 'driver' ou 'passenger'
    """
    identity_col = f"{user_type}_identity_key"
    
    # Données filtrées pour ce type d'utilisateur
    df_filtered = df.filter(
        pl.col("first_trip_datetime") >= datetime(2024, 9, 1, tzinfo=ZoneInfo("GMT")),
        pl.col("first_trip_datetime") <= datetime(2025, 7, 31, tzinfo=ZoneInfo("GMT")),  # Coupure juillet 2025
        pl.col(identity_col).is_not_null()
    )
    
    # Analyser l'activité mensuelle
    user_stats = (
        df_filtered
        .group_by(identity_col)
        .agg([
            # Premier trajet pour définir la fenêtre de suivi
            pl.col("start_datetime").min().alias("first_trip"),
            # Mois actifs (mois où l'utilisateur a fait au moins un trajet)
            pl.col("start_datetime").dt.truncate("1mo").n_unique().alias("active_months"),
            # Total de trajets uniques
            pl.concat_str(pl.col("operator_id"), pl.lit("-"), pl.col("operator_trip_id"))
            .n_unique()
            .alias("total_trajets")
        ])
        .with_columns([
            # One shot: 1 seul mois actif ET 1 seul trajet
            (
                (pl.col("active_months") == 1) &
                (pl.col("total_trajets") == 1)
            ).alias(f"one_shot_{user_type}"),
            
            # Ten shots: exactement 10 trajets
            (pl.col("total_trajets") == 10).alias(f"ten_shots_{user_type}"),
            
            # Regular: au moins un trajet par mois sur 4 mois (consécutifs ou non)
            (pl.col("active_months") >= 4).alias(f"regular_{user_type}")
        ])
        .select([
            identity_col, 
            f"one_shot_{user_type}", 
            f"ten_shots_{user_type}",
            f"regular_{user_type}", 
            "active_months",
            "total_trajets"
        ])
    )
    
    return user_stats

# Créer les flags pour conducteurs et passagers
df_driver_flags = create_user_flags(df_journeys_raw, "driver")
df_passenger_flags = create_user_flags(df_journeys_raw, "passenger")

# Joindre au dataset principal avec gestion des nulls
df_with_flags = (
    df_journeys_raw
    .join(df_driver_flags, on="driver_identity_key", how="left")
    .join(df_passenger_flags, on="passenger_identity_key", how="left")
    .with_columns([
        # Remplacer les nulls par False pour les utilisateurs avec identity_key valide
        pl.when(pl.col("driver_identity_key").is_not_null())
        .then(pl.col("one_shot_driver").fill_null(False))
        .otherwise(None)
        .alias("one_shot_driver"),
        
        pl.when(pl.col("driver_identity_key").is_not_null())
        .then(pl.col("ten_shots_driver").fill_null(False))
        .otherwise(None)
        .alias("ten_shots_driver"),
        
        pl.when(pl.col("driver_identity_key").is_not_null())
        .then(pl.col("regular_driver").fill_null(False))
        .otherwise(None)
        .alias("regular_driver"),
        
        pl.when(pl.col("passenger_identity_key").is_not_null())
        .then(pl.col("one_shot_passenger").fill_null(False))
        .otherwise(None)
        .alias("one_shot_passenger"),
        
        pl.when(pl.col("passenger_identity_key").is_not_null())
        .then(pl.col("ten_shots_passenger").fill_null(False))
        .otherwise(None)
        .alias("ten_shots_passenger"),
        
        pl.when(pl.col("passenger_identity_key").is_not_null())
        .then(pl.col("regular_passenger").fill_null(False))
        .otherwise(None)
        .alias("regular_passenger")
    ])
)

# Vérifications
print("Répartition CONDUCTEURS:")
print(df_driver_flags.group_by(["one_shot_driver", "ten_shots_driver", "regular_driver"]).agg(pl.len().alias("count")))

print("\nRépartition PASSAGERS:")
print(df_passenger_flags.group_by(["one_shot_passenger", "ten_shots_passenger", "regular_passenger"]).agg(pl.len().alias("count")))

# Statistiques détaillées par catégorie
print("\nStatistiques détaillées CONDUCTEURS:")
print(df_driver_flags.select([
    "active_months", 
    "total_trajets", 
    "one_shot_driver", 
    "ten_shots_driver", 
    "regular_driver"
]).describe())

print("\nStatistiques détaillées PASSAGERS:")
print(df_passenger_flags.select([
    "active_months", 
    "total_trajets", 
    "one_shot_passenger", 
    "ten_shots_passenger", 
    "regular_passenger"
]).describe())

## Acquisition conducteur one-shot

In [113]:
from plotly.subplots import make_subplots

df_with_categories = df_with_flags.with_columns([
    pl.when(pl.col("one_shot_driver") == True)
    .then(pl.lit("One-shot"))
    .when((pl.col("ten_shots_driver") == True) & (pl.col("regular_driver") == False))
    .then(pl.lit("Ten-shot"))
    .when(pl.col("regular_driver") == True)
    .then(pl.lit("Régulier"))
    .when((pl.col("one_shot_driver") == False) & (pl.col("ten_shots_driver") == False) & (pl.col("regular_driver") == False))
    .then(pl.lit("Occasionnel"))
    .otherwise(None)  # Pour les null (pas d'identity_key)
    .alias("driver_category")
])

# Données pour le graphique
chart_data = (
    df_with_categories.filter(
        pl.col("first_trip_datetime") >= datetime(2024, 9, 1, tzinfo=ZoneInfo("GMT")),
        pl.col("driver_category").is_not_null()  # Exclure les null
    )
    .group_by([
        pl.col("first_trip_datetime").dt.truncate("1w").alias("week"),
        "driver_category"
    ])
    .agg(pl.len().alias("count"))
    .sort(["week", "driver_category"])
)

fig_new_drivers_categories_by_week = px.bar(
    chart_data,
    x="week",
    y="count",
    color="driver_category",
    labels={**labels_map, "count": "Nombre de nouveaux conducteurs", "driver_category": "Type de conducteur"},
    template="simple_white",
    title="Evolution de l'acquisition des conducteurs par catégorie",
    # barmode="overlay",
    color_discrete_map = {
        "One-shot": "#ff7f0e",     # Orange
        "Ten-shot": "#ffb366",     # Orange plus clair (proche de One-shot)
        "Régulier": "#2ca02c",     # Vert
        "Occasionnel": "#1f77b4"   # Bleu
    }
)

fig_new_drivers_categories_by_week.show()
fig_new_drivers_categories_by_week.write_html(OUTPUT_PATH / "fig_new_drivers_categories_by_week.html")
fig_new_drivers_categories_by_week.write_image(
    OUTPUT_PATH / "fig_new_drivers_categories_by_week.svg", width=1280, height=720
)

## Distance moyenne

In [114]:
def add_user_type_traces(fig, df_data, user_type_col, y_metric, 
                        user_types=None, color_map=None, 
                        row=1, col=1, showlegend=True, 
                        x_col="week", mode='lines+markers'):
    """
    Ajoute des traces par type d'utilisateur à un graphique plotly
    """
    
    # Valeurs par défaut
    if user_types is None:
        user_types = ["One-shot", "Occasionnel", "Régulier", "Ten-shots"]
    
    if color_map is None:
        color_map = {
            "One-shot": "#ff7f0e",      # Orange
            "Ten-shots": "#ffb366",     # Orange plus clair (proche de One-shot)
            "Régulier": "#2ca02c",      # Vert  
            "Occasionnel": "#1f77b4"   # Bleu
        }
    
    # Ajouter les traces
    for user_type in user_types:
        df_subset = df_data.filter(pl.col(user_type_col) == user_type).to_pandas()
        
        if len(df_subset) > 0:
            fig.add_trace(
                go.Scatter(
                    x=df_subset[x_col],
                    y=df_subset[y_metric],
                    mode=mode,
                    name=user_type,
                    line=dict(color=color_map[user_type]),
                    marker=dict(color=color_map[user_type]),
                    legendgroup=user_type,
                    showlegend=showlegend
                ),
                row=row, col=col
            )

def create_user_comparison_plot(df_drivers, df_passengers, y_metric, filename,
                               title=None, y_label=None, x_col="week", x_label=None,
                               user_types=None, color_map=None, mode='lines+markers',
                               output_path=None, labels_map=None,
                               template="simple_white", height=500, width=1200):
    """
    Crée un graphique complet avec conducteurs et passagers, l'affiche et le sauvegarde
    
    Parameters:
    - df_drivers, df_passengers: DataFrames polars avec les données
    - y_metric: Métrique à afficher sur l'axe y
    - filename: Nom du fichier (sans extension)
    - title: Titre du graphique (optionnel)
    - y_label, x_label: Labels des axes (optionnels)
    - user_types: Types d'utilisateurs à afficher
    - color_map: Couleurs personnalisées
    - mode: Mode d'affichage ('lines+markers', 'lines', etc.)
    - output_path: Chemin de sauvegarde (défaut: OUTPUT_PATH)
    - labels_map: Dictionnaire des labels
    - template, height, width: Paramètres de style
    """
    
    # Valeurs par défaut
    if output_path is None:
        output_path = OUTPUT_PATH
    
    if title is None:
        title = f"Évolution de {y_metric} par type d'utilisateur"
    
    if x_label is None:
        x_label = labels_map.get(x_col, x_col.capitalize()) if labels_map else x_col.capitalize()
    
    if y_label is None:
        y_label = labels_map.get(y_metric, y_metric) if labels_map else y_metric
    
    # Créer la figure
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=["Conducteurs", "Passagers"],
        shared_yaxes=True,
        horizontal_spacing=0.1
    )
    
    # Ajouter les traces
    add_user_type_traces(
        fig=fig, 
        df_data=df_drivers, 
        user_type_col="driver_type",
        y_metric=y_metric,
        user_types=user_types,
        color_map=color_map,
        row=1, col=1, 
        showlegend=True,
        x_col=x_col,
        mode=mode
    )
    
    add_user_type_traces(
        fig=fig, 
        df_data=df_passengers, 
        user_type_col="passenger_type",
        y_metric=y_metric,
        user_types=user_types,
        color_map=color_map,
        row=1, col=2, 
        showlegend=False,
        x_col=x_col,
        mode=mode
    )
    
    # Mise en forme
    fig.update_layout(
        title=title,
        template=template,
        height=height,
        width=width,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    
    # Labels des axes
    fig.update_xaxes(title_text=x_label)
    fig.update_yaxes(title_text=y_label, col=1)
    
    fig.show()
    
    html_path = output_path / f"{filename}.html"
    svg_path = output_path / f"{filename}.svg"
    
    fig.write_html(html_path)
    fig.write_image(svg_path, width=1280, height=720)

    
    return fig

In [115]:
df_temporal_analysis = df_with_flags.with_columns([
    pl.when(pl.col("one_shot_driver") == True).then(pl.lit("One-shot"))
    .when(pl.col("regular_driver") == True).then(pl.lit("Régulier"))  
    .when(pl.col("driver_identity_key").is_not_null()).then(pl.lit("Occasionnel"))
    .otherwise(None)
    .alias("driver_type"),
    
    pl.when(pl.col("one_shot_passenger") == True).then(pl.lit("One-shot"))
    .when(pl.col("regular_passenger") == True).then(pl.lit("Régulier"))
    .when(pl.col("passenger_identity_key").is_not_null()).then(pl.lit("Occasionnel"))
    .otherwise(None)
    .alias("passenger_type")
])

df_drivers_by_week = (
    df_temporal_analysis.filter(pl.col("driver_identity_key").is_not_null())
    .group_by([
        pl.col("start_datetime").dt.truncate("1w").alias("week"),
        "driver_type"
    ])
    .agg(agg_expressions)  
    .sort(pl.col("week"))
)

df_passengers_by_week = (
    df_temporal_analysis.filter(pl.col("passenger_identity_key").is_not_null())
    .group_by([
        pl.col("start_datetime").dt.truncate("1w").alias("week"),
        "passenger_type"
    ])
    .agg(agg_expressions) 
    .sort(pl.col("week"))
)

color_map = {
    "One-shot": "#ff7f0e",      # Orange
    "Régulier": "#2ca02c",      # Vert  
    "Occasionnel": "#1f77b4"   # Bleu
}

fig_journeys_avg_distance_by_user_type = create_user_comparison_plot(
    df_drivers=df_drivers_by_week,
    df_passengers=df_passengers_by_week,
    y_metric="distance_avg",
    filename="fig_journeys_avg_distance_by_user_type",
    title="Distance moyenne par type d'utilisateur",
    y_label="Distance moyenne (km)",
    labels_map=labels_map
)


In [116]:
fig_journeys_avg_distance_by_user_type_rm_onehsot = create_user_comparison_plot(
    df_drivers=df_drivers_by_week,
    df_passengers=df_passengers_by_week,
    y_metric="distance_avg",
    filename="fig_journeys_avg_distance_by_user_type_rm_onehsot",
    title="Distance moyenne par type d'utilisateur",
    y_label="Distance moyenne (km)",
    user_types=["Occasionnel", "Régulier"], 
    labels_map=labels_map
)


In [117]:
df_temporal_analysis = df_with_flags.with_columns([
    pl.when(pl.col("one_shot_driver") == True).then(pl.lit("One-shot"))
    .when(pl.col("ten_shots_driver") == True).then(pl.lit("Ten-shots"))
    .when(pl.col("regular_driver") == True).then(pl.lit("Régulier"))  
    .when(pl.col("driver_identity_key").is_not_null()).then(pl.lit("Occasionnel"))
    .otherwise(None)
    .alias("driver_type"),
    
    pl.when(pl.col("one_shot_passenger") == True).then(pl.lit("One-shot"))
    .when(pl.col("ten_shots_passenger") == True).then(pl.lit("Ten-shots"))
    .when(pl.col("regular_passenger") == True).then(pl.lit("Régulier"))
    .when(pl.col("passenger_identity_key").is_not_null()).then(pl.lit("Occasionnel"))
    .otherwise(None)
    .alias("passenger_type")
])

df_drivers_by_week = (
    df_temporal_analysis.filter(pl.col("driver_identity_key").is_not_null())
    .group_by([
        pl.col("start_datetime").dt.truncate("1w").alias("week"),
        "driver_type"
    ])
    .agg(agg_expressions) 
    .sort(pl.col("week"))
)

df_passengers_by_week = (
    df_temporal_analysis.filter(pl.col("passenger_identity_key").is_not_null())
    .group_by([
        pl.col("start_datetime").dt.truncate("1w").alias("week"),
        "passenger_type"
    ])
    .agg(agg_expressions)  
    .sort(pl.col("week"))
)

color_map = {
    "One-shot": "#ff7f0e",      # Orange
    "Ten-shots": "#ffb366",     # Orange plus clair (proche de One-shot)
    "Régulier": "#2ca02c",      # Vert  
    "Occasionnel": "#1f77b4"   # Bleu
}

fig_journeys_avg_distance_by_user_type = create_user_comparison_plot(
    df_drivers=df_drivers_by_week,
    df_passengers=df_passengers_by_week,
    y_metric="distance_avg",
    filename="fig_journeys_avg_distance_by_user_type",
    title="Distance moyenne des trajets par semaine par type d'utilisateur",
    y_label="Distance moyenne (km)",
    labels_map=labels_map
)


# Distribution des distances

## Tout trajet confondu

In [118]:
fig_distance_distrib = px.histogram(
    df_with_flags.with_columns((pl.col("distance") / 1000).alias("distance_km")).filter(pl.col("distance_km")<100),
    x="distance_km",
    nbins=100,
    template="simple_white",
    labels=labels_map,
    title="Distribution de la distance effectuée par journey",
    height=500,
)

fig_distance_distrib.show()

fig_distance_distrib.write_html(
    "outputs_idfm/fig_distance_distrib.html"
)
fig_distance_distrib.write_image(
    "outputs_idfm/fig_distance_distrib.svg", width=1280, height=720
)

## Utilisateurs réguliers, one shot et autre

In [ ]:
from plotly.subplots import make_subplots

df_plot = df_with_flags.with_columns([
    (pl.col("distance") / 1000).alias("distance_km"),  # Conversion en km
    # Type conducteur
    pl.when(pl.col("one_shot_driver") == True).then(pl.lit("One-shot"))
    .when((pl.col("ten_shots_driver") == True) & (pl.col("regular_driver") == False)).then(pl.lit("Ten-shot"))
    .when(pl.col("regular_driver") == True).then(pl.lit("Régulier"))
    .when(pl.col("driver_identity_key").is_not_null()).then(pl.lit("Occasionnel"))
    .otherwise(None)
    .alias("driver_type"),
    # Type passager
    pl.when(pl.col("one_shot_passenger") == True).then(pl.lit("One-shot"))
    .when((pl.col("ten_shots_passenger") == True) & (pl.col("regular_passenger") == False)).then(pl.lit("Ten-shot"))
    .when(pl.col("regular_passenger") == True).then(pl.lit("Régulier"))
    .when(pl.col("passenger_identity_key").is_not_null()).then(pl.lit("Occasionnel"))
    .otherwise(None)
    .alias("passenger_type")
]).to_pandas()

# Couleurs consistantes
color_map = {
    "One-shot": "#ff7f0e",     # Orange
    "Ten-shot": "#ffb366",     # Orange plus clair (proche de One-shot)
    "Régulier": "#2ca02c",     # Vert
    "Occasionnel": "#1f77b4"   # Bleu
}

# Créer les subplots
fig_distance_distrib = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Conducteurs", "Passagers"],
    shared_yaxes=True,
    horizontal_spacing=0.1
)

# Conducteurs
df_drivers = df_plot[df_plot["driver_type"].notna()]
for user_type in ["One-shot", "Ten-shot", "Occasionnel", "Régulier"]:
    data = df_drivers[df_drivers["driver_type"] == user_type]["distance_km"]
    fig_distance_distrib.add_trace(
        go.Histogram(
            x=data,
            xbins=dict(
                start=0,
                end=100,
                size=1  # 100 bins de 1 km chacun
            ),
            name=user_type,
            histnorm='percent',
            marker_color=color_map[user_type],
            opacity=0.7,
            legendgroup=user_type,
            showlegend=True
        ),
        row=1, col=1
    )

# Passagers
df_passengers = df_plot[df_plot["passenger_type"].notna()]
for user_type in ["One-shot", "Ten-shot", "Occasionnel", "Régulier"]:
    data = df_passengers[df_passengers["passenger_type"] == user_type]["distance_km"]
    fig_distance_distrib.add_trace(
        go.Histogram(
            x=data,
            xbins=dict(
                start=0,
                end=100,
                size=1  # 100 bins de 1 km chacun
            ),
            name=user_type,
            histnorm='percent',
            marker_color=color_map[user_type],
            opacity=0.7,
            legendgroup=user_type,
            showlegend=False  # Éviter duplication dans légende
        ),
        row=1, col=2
    )

# Mise en forme
fig_distance_distrib.update_layout(
    title="Distribution de la distance effectuée par journey par type d'utilisateur",
    template="simple_white",
    height=500,
    width=1000,
    barmode='overlay',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

# Fixer le range et les labels
fig_distance_distrib.update_xaxes(
    range=[0, 100],
    title_text=labels_map.get("distance_km", "Distance (km)")
)
fig_distance_distrib.update_yaxes(
    title_text="% de trajets",
    col=1
)

# Afficher
fig_distance_distrib.show()

# Sauvegarder
fig_distance_distrib.write_html(
    "outputs_idfm/fig_distance_distrib_by_user_type.html"
)
fig_distance_distrib.write_image(
    "outputs_idfm/fig_distance_distrib_by_user_type.svg",
    width=1280,
    height=720
)